# Domain 1: Agentic Architecture & Orchestration
## Task Statement 1.3: Configure Subagent Invocation, Context Passing, and Spawning
**Certification Weight**: 27% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to spawn subagents using the Agent SDK `Task` tool, trigger parallel execution in a single model turn, pass explicit goal-oriented context, and preserve citation attribution across subagent boundaries.

---

## 🏗️ Architecture: Parallel Spawning & Attribution Flow

```mermaid
flowchart TD
    Coordinator["Coordinator Turn<br>(claude-sonnet-4-5)"] --> MultiCall["Single Turn Emits Multiple Tool Calls:<br>Task(subagent='search', query='A')<br>Task(subagent='search', query='B')"]
    
    MultiCall --> ExecPool["Parallel SDK Execution Pool"]
    
    ExecPool --> WorkerA["Subagent Worker A<br>(Fresh Context)"]
    ExecPool --> WorkerB["Subagent Worker B<br>(Fresh Context)"]
    
    WorkerA --> ResultA["Result Payload A:<br>content + metadata(url, title, date)"]
    WorkerB --> ResultB["Result Payload B:<br>content + metadata(url, title, date)"]
    
    ResultA --> Collector["Collate Tool Results"]
    ResultB --> Collector
    Collector --> NextTurn["Coordinator Next Turn:<br>Synthesize with full attribution"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Concept | Correct Implementation | Exam Trap to Reject |
|---|---|---|
| **Subagent Spawning Mechanism** | Use the SDK `Task` tool with subagent specification in `allowedTools`. | **TRAP**: Calling arbitrary shell scripts or non-standard external endpoints directly. |
| **Spawning Latency Optimization** | **Parallel Spawning**: Emit multiple `Task` tool use blocks in a **single** coordinator turn. | **TRAP**: Emitting one `Task` call per turn sequentially, multiplying round-trip latency. |
| **Context Transmission** | **Explicit Context Passing**: Provide complete instructions and input data in the subagent prompt. | **TRAP**: Assuming the spawned subagent inherits parent conversation memory or global session state. |
| **Source Attribution** | Maintain structured separation between `content` and `metadata` (source URL, author, timestamp). | **TRAP**: Merging raw text with citations flattened into unparseable prose. |
| **Prompt Delegation Style** | **Goal-oriented**: Define desired outcome and quality constraints (e.g. "Find API rate limits and SLAs"). | **TRAP**: Over-specifying rigid keystroke-level procedural commands that defeat agent flexibility. |

---

## 💡 Architectural Decision Matrix

| Dimension | Recommended Pattern | Rationale |
|---|---|---|
| **Spawning Topology** | Fan-out in single turn, gather in next turn | Minimizes Claude API network hops and reduces total wall-clock latency for research tasks. |
| **Attribution Contract** | Typed JSON schema with explicit metadata fields | Prevents source hallucination during final synthesis and guarantees auditability. |
| **Subagent Scope** | Bounded single-responsibility tasks | Keeps subagent prompts concise and prevents tool search saturation. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Offline Setup
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 1.3 Subagent Invocation & Spawning.")

### 📋 Step 2: Agent Definitions & The Task Tool Schema
The `Task` tool declaration allows Claude to invoke registered subagents dynamically with dedicated prompts and metadata.

In [ ]:
# Step 2: Subagent Registry & Task Tool Schema
@dataclass
class AgentDefinition:
    name: str
    description: str
    system_prompt: str
    allowed_tools: List[str]

SUBAGENT_REGISTRY = {
    "web_researcher": AgentDefinition(
        name="web_researcher",
        description="Researches technical documentation and articles.",
        system_prompt="You are a precise technical researcher.",
        allowed_tools=["fetch_url", "search"],
    ),
    "code_analyzer": AgentDefinition(
        name="code_analyzer",
        description="Inspects source code repositories, classes, and syntax.",
        system_prompt="You are an expert static code analyst.",
        allowed_tools=["grep", "read_file"],
    ),
}

TASK_TOOL = {
    "name": "Task",
    "description": (
        "Spawns an isolated subagent to perform a targeted task. "
        "Multiple Task calls may be emitted in a single turn for concurrent execution."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "subagent_type": {
                "type": "string",
                "enum": ["web_researcher", "code_analyzer"],
                "description": "Registered subagent type to instantiate.",
            },
            "prompt": {
                "type": "string",
                "description": "Complete, self-contained prompt for the subagent.",
            },
            "task_metadata": {
                "type": "object",
                "properties": {
                    "task_id": {"type": "string"},
                    "expected_output_format": {"type": "string"},
                },
                "required": ["task_id"],
            },
        },
        "required": ["subagent_type", "prompt", "task_metadata"],
    },
}

print(" Task Tool schema and subagent registry defined.")

### ⚡ Step 3: Parallel Execution Engine
Simulates executing multiple subagent tasks spawned concurrently by the coordinator.

In [ ]:
# Step 3: Subagent Execution Engine
def execute_subagent_task(subagent_type: str, prompt: str, metadata: Dict[str, Any]) -> Dict[str, Any]:
    """Execute an individual subagent task with metadata/content separation."""
    if subagent_type not in SUBAGENT_REGISTRY:
        return {"status": "error", "error": f"Unknown subagent: {subagent_type}"}

    if subagent_type == "web_researcher":
        return {
            "task_id": metadata.get("task_id"),
            "subagent_type": subagent_type,
            "status": "completed",
            "findings": [
                {
                    "content": "Claude Agent SDK uses the Task tool for spawning child agents.",
                    "metadata": {
                        "source_url": "https://docs.anthropic.com/agents",
                        "confidence": 0.98,
                    },
                }
            ],
        }
    elif subagent_type == "code_analyzer":
        return {
            "task_id": metadata.get("task_id"),
            "subagent_type": subagent_type,
            "status": "completed",
            "findings": [
                {
                    "content": "Found 2 occurrences of allowedTools containing 'Task'.",
                    "metadata": {
                        "file": "src/coordinator.ts",
                        "lines": [42, 88],
                    },
                }
            ],
        }
    return {"status": "error", "error": "Unhandled type"}

def run_parallel_spawning_coordinator(user_query: str, client: Any) -> Dict[str, Any]:
    """Simulates coordinator receiving a prompt and issuing parallel Task calls."""
    # Turn 1: Model emits parallel tool calls
    resp = client.messages.create(
        model="claude-haiku-4-5",
        messages=[{"role": "user", "content": user_query}],
        tools=[TASK_TOOL],
    )

    completed_tasks = []
    if resp.stop_reason == "tool_use":
        for block in resp.content:
            if block.type == "tool_use" and block.name == "Task":
                result = execute_subagent_task(
                    block.input["subagent_type"],
                    block.input["prompt"],
                    block.input["task_metadata"],
                )
                completed_tasks.append(result)

    return {"spawned_count": len(completed_tasks), "tasks": completed_tasks}

print(" Parallel subagent dispatcher ready.")

### 🧪 Step 4: Offline Simulation Walkthrough
We configure a mock response emitting **two** `Task` calls in a single turn to demonstrate parallel subagent spawning.

In [ ]:
# Step 4: Parallel Spawning Simulation
# Mock coordinator issuing TWO concurrent Task calls in Turn 1
parallel_resp = MockMessageResponse(
    content=[
        MockContentBlock(
            block_type="tool_use",
            name="Task",
            tool_id="call_task_web_01",
            input_data={
                "subagent_type": "web_researcher",
                "prompt": "Search for Agent SDK Task tool guidelines.",
                "task_metadata": {"task_id": "T-WEB-001"},
            },
        ),
        MockContentBlock(
            block_type="tool_use",
            name="Task",
            tool_id="call_task_code_02",
            input_data={
                "subagent_type": "code_analyzer",
                "prompt": "Locate allowedTools definitions in repository.",
                "task_metadata": {"task_id": "T-CODE-002"},
            },
        ),
    ],
    stop_reason="tool_use",
)

mock_client = MockClaudeClient([parallel_resp])
sim_res = run_parallel_spawning_coordinator("Analyze Task tool usage", mock_client)

print(f"Spawned Subagent Tasks: {sim_res['spawned_count']}")
for t in sim_res["tasks"]:
    print(f"  - Task ID: {t['task_id']}, Type: {t['subagent_type']}, Status: {t['status']}")

### 🔬 Step 5: Verification Assertions
We assert that:
1. Two distinct subagents were invoked in parallel within the same turn.
2. Output payloads cleanly separate substantive content from tracking metadata.

In [ ]:
# Step 5: Verification Assertions
assert sim_res["spawned_count"] == 2, "Expected 2 parallel subagent spawns"

web_task = next(t for t in sim_res["tasks"] if t["subagent_type"] == "web_researcher")
assert web_task["task_id"] == "T-WEB-001"
finding = web_task["findings"][0]
assert "content" in finding
assert "metadata" in finding
assert "source_url" in finding["metadata"]

print(" All verification assertions passed! Parallel spawning verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Spawning subagents requires `Task` in `allowedTools`.
- **Rule 2**: Emitting multiple `Task` calls in a single turn achieves true concurrency (reducing latency 40-60%).
- **Rule 3**: Separate content from metadata (`source_url`, `confidence`, `timestamp`) to preserve provenance.